# 02 Baseline Model

本Notebook用于建立恒星类型预测项目的基准模型。首先使用DummyClassifier构造最低基准线，然后使用RandomForest和ExtraTrees建立正式Baseline，并通过accuracy、balanced accuracy、macro F1、classification report和confusion matrix进行评估。

In [ ]:
# 导入库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)


In [ ]:
# 设置路径
DATA_DIR = Path("../data/raw")

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("train shape:", train.shape)
print("test shape:", test.shape)
print("sample_submission shape:", sample_submission.shape)

In [ ]:
# 读取数据并识别目标列
possible_target_cols = [col for col in train.columns if col not in test.columns]

print("Possible target columns:", possible_target_cols)

if len(possible_target_cols) != 1:
    raise ValueError(f"目标列识别异常，请手动检查：{possible_target_cols}")

target_col = possible_target_cols[0]
print("Target column:", target_col)

In [ ]:
# 定义特征和目标变量
id_cols = ["id"] if "id" in train.columns else []

X = train.drop(columns=id_cols + [target_col])
y = train[target_col]

X_test = test.drop(columns=id_cols, errors="ignore")

print("X shape:", X.shape)
print("y shape:", y.shape)
print("X_test shape:", X_test.shape)

In [ ]:
# 识别数值型和类别型特征
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numerical columns:", num_cols)
print("Categorical columns:", cat_cols)

In [ ]:
# 划分训练集和验证集，保持类别分布一致
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training target distribution:")
print(y_train.value_counts(normalize=True))

print("\nValidation target distribution:")
print(y_valid.value_counts(normalize=True))

In [ ]:
# DummyClassifier模型
# ============================================================
# 1.设置Dummy模型结果保存目录
# ============================================================

MODEL_NAME = "dummy"

REPORT_DIR = Path("../reports/model/baseline") / MODEL_NAME
TABLE_DIR = REPORT_DIR / "tables"
FIGURE_DIR = REPORT_DIR / "figures"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("REPORT_DIR:", REPORT_DIR.resolve())
print("TABLE_DIR:", TABLE_DIR.resolve())
print("FIGURE_DIR:", FIGURE_DIR.resolve())

# ============================================================
# 2.训练DummyClassifier
# ============================================================

dummy_model = DummyClassifier(strategy="most_frequent")

dummy_model.fit(X_train, y_train)

dummy_pred = dummy_model.predict(X_valid)


# ============================================================
# 3.计算评价指标
# ============================================================

dummy_acc = accuracy_score(y_valid, dummy_pred)
dummy_balanced_acc = balanced_accuracy_score(y_valid, dummy_pred)
dummy_macro_f1 = f1_score(y_valid, dummy_pred, average="macro")

dummy_report = classification_report(
    y_valid,
    dummy_pred,
    zero_division=0
)

print("Dummy Classifier Results")
print("=" * 60)
print(f"Accuracy: {dummy_acc:.6f}")
print(f"Balanced Accuracy: {dummy_balanced_acc:.6f}")
print(f"Macro F1: {dummy_macro_f1:.6f}")
print("\nClassification Report:")
print(dummy_report)


# ============================================================
# 4.保存总体评价指标
# ============================================================

dummy_metrics = pd.DataFrame({
    "model": ["DummyClassifier_most_frequent"],
    "accuracy": [dummy_acc],
    "balanced_accuracy": [dummy_balanced_acc],
    "macro_f1": [dummy_macro_f1]
})

metrics_path = TABLE_DIR / "dummy_classifier_metrics.csv"
dummy_metrics.to_csv(metrics_path, index=False, encoding="utf-8-sig")

print("Saved metrics:", metrics_path)


# ============================================================
# 5.保存classification report为txt
# ============================================================

report_txt_path = TABLE_DIR / "dummy_classifier_report.txt"

with open(report_txt_path, "w", encoding="utf-8") as f:
    f.write("Dummy Classifier Baseline Report\n")
    f.write("=" * 60 + "\n")
    f.write("Strategy: most_frequent\n")
    f.write(f"Accuracy: {dummy_acc:.6f}\n")
    f.write(f"Balanced Accuracy: {dummy_balanced_acc:.6f}\n")
    f.write(f"Macro F1: {dummy_macro_f1:.6f}\n\n")
    f.write("Classification Report:\n")
    f.write(dummy_report)

print("Saved report txt:", report_txt_path)


# ============================================================
# 6.保存classification report为csv
# ============================================================

dummy_report_dict = classification_report(
    y_valid,
    dummy_pred,
    zero_division=0,
    output_dict=True
)

dummy_report_df = pd.DataFrame(dummy_report_dict).T

report_csv_path = TABLE_DIR / "dummy_classifier_classification_report.csv"
dummy_report_df.to_csv(report_csv_path, encoding="utf-8-sig")

print("Saved report csv:", report_csv_path)


# ============================================================
# 7.绘制并保存混淆矩阵图片
# ============================================================

labels = sorted(y_valid.unique())

dummy_cm = confusion_matrix(
    y_valid,
    dummy_pred,
    labels=labels
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=dummy_cm,
    display_labels=labels
)

disp.plot(values_format="d")

plt.title("Confusion Matrix - Dummy Classifier")
plt.savefig(
    FIGURE_DIR / "dummy_classifier_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

print("Saved confusion matrix figure:", FIGURE_DIR / "dummy_classifier_confusion_matrix.png")


# ============================================================
# 8.保存混淆矩阵为csv
# ============================================================

dummy_cm_df = pd.DataFrame(
    dummy_cm,
    index=[f"true_{label}" for label in labels],
    columns=[f"pred_{label}" for label in labels]
)

cm_csv_path = TABLE_DIR / "dummy_classifier_confusion_matrix.csv"
dummy_cm_df.to_csv(cm_csv_path, encoding="utf-8-sig")

print("Saved confusion matrix csv:", cm_csv_path)

In [ ]:
# 补充导入库
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# 设置Random Forest模型结果保存目录
MODEL_NAME = "random_forest"

REPORT_DIR = Path("../reports/model/baseline") / MODEL_NAME
TABLE_DIR = REPORT_DIR / "tables"
FIGURE_DIR = REPORT_DIR / "figures"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("REPORT_DIR:", REPORT_DIR.resolve())
print("TABLE_DIR:", TABLE_DIR.resolve())
print("FIGURE_DIR:", FIGURE_DIR.resolve())

In [ ]:
# 建立RandomForest Pipeline
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [col for col in X.columns if col not in num_cols]

print("Numerical columns:", num_cols)
print("Categorical columns:", cat_cols)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ]
)

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", rf_model)
    ]
)

rf_pipeline.fit(X_train, y_train)

In [ ]:
#评估RandomForest
rf_pred = rf_pipeline.predict(X_valid)

rf_acc = accuracy_score(y_valid, rf_pred)
rf_balanced_acc = balanced_accuracy_score(y_valid, rf_pred)
rf_macro_f1 = f1_score(y_valid, rf_pred, average="macro")

rf_report = classification_report(
    y_valid,
    rf_pred,
    zero_division=0
)

print("Random Forest Results")
print("=" * 60)
print(f"Accuracy: {rf_acc:.6f}")
print(f"Balanced Accuracy: {rf_balanced_acc:.6f}")
print(f"Macro F1: {rf_macro_f1:.6f}")
print("\nClassification Report:")
print(rf_report)

In [ ]:
#保存RandomForest总体评价指标
rf_metrics = pd.DataFrame({
    "model": ["RandomForest_baseline"],
    "accuracy": [rf_acc],
    "balanced_accuracy": [rf_balanced_acc],
    "macro_f1": [rf_macro_f1]
})

metrics_path = TABLE_DIR / "random_forest_metrics.csv"
rf_metrics.to_csv(metrics_path, index=False, encoding="utf-8-sig")

report_txt_path = TABLE_DIR / "random_forest_report.txt"

with open(report_txt_path, "w", encoding="utf-8") as f:
    f.write("Random Forest Baseline Report\n")
    f.write("=" * 60 + "\n")
    f.write(f"Accuracy: {rf_acc:.6f}\n")
    f.write(f"Balanced Accuracy: {rf_balanced_acc:.6f}\n")
    f.write(f"Macro F1: {rf_macro_f1:.6f}\n\n")
    f.write("Classification Report:\n")
    f.write(rf_report)

rf_report_dict = classification_report(
    y_valid,
    rf_pred,
    zero_division=0,
    output_dict=True
)

rf_report_df = pd.DataFrame(rf_report_dict).T
rf_report_df.to_csv(
    TABLE_DIR / "random_forest_classification_report.csv",
    encoding="utf-8-sig"
)

print("Saved RandomForest results.")

In [ ]:
#保存RandomForest混淆矩阵
labels = sorted(y_valid.unique())

rf_cm = confusion_matrix(
    y_valid,
    rf_pred,
    labels=labels
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=rf_cm,
    display_labels=labels
)

disp.plot(values_format="d")

plt.title("Confusion Matrix - Random Forest")
plt.savefig(
    FIGURE_DIR / "random_forest_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

rf_cm_df = pd.DataFrame(
    rf_cm,
    index=[f"true_{label}" for label in labels],
    columns=[f"pred_{label}" for label in labels]
)

rf_cm_df.to_csv(
    TABLE_DIR / "random_forest_confusion_matrix.csv",
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# 保存RandomForest特征重要性
# ============================================================

# 1.获取One-Hot之后的特征名
preprocessor = rf_pipeline.named_steps["preprocessor"]
rf_model = rf_pipeline.named_steps["model"]

num_feature_names = num_cols

cat_encoder = preprocessor.named_transformers_["cat"]
cat_feature_names = cat_encoder.get_feature_names_out(cat_cols).tolist()

all_feature_names = num_feature_names + cat_feature_names

# 2.提取特征重要性
feature_importance = pd.DataFrame({
    "feature": all_feature_names,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

display(feature_importance.head(30))

# 3.保存为CSV
feature_importance_path = TABLE_DIR / "random_forest_feature_importance.csv"
feature_importance.to_csv(feature_importance_path, index=False, encoding="utf-8-sig")

print("Saved feature importance:", feature_importance_path)

# 4.绘制Top 20特征重要性图
top_n = 20
top_features = feature_importance.head(top_n).sort_values("importance")

plt.figure(figsize=(10, 8))
plt.barh(top_features["feature"], top_features["importance"])
plt.title("Top 20 Feature Importance - Random Forest")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()

feature_importance_fig_path = FIGURE_DIR / "random_forest_feature_importance_top20.png"
plt.savefig(feature_importance_fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved feature importance figure:", feature_importance_fig_path)

In [ ]:
# ============================================================
# ExtraTreesClassifier Baseline
# 保存路径：reports/model/baseline/extra_trees/
# ============================================================

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# ============================================================
# 1.设置ExtraTrees结果保存目录
# ============================================================

MODEL_NAME = "extra_trees"

REPORT_DIR = Path("../reports/model/baseline") / MODEL_NAME
TABLE_DIR = REPORT_DIR / "tables"
FIGURE_DIR = REPORT_DIR / "figures"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("REPORT_DIR:", REPORT_DIR.resolve())
print("TABLE_DIR:", TABLE_DIR.resolve())
print("FIGURE_DIR:", FIGURE_DIR.resolve())


# ============================================================
# 2.识别数值变量和分类变量
# ============================================================

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [col for col in X.columns if col not in num_cols]

print("Numerical columns:", num_cols)
print("Categorical columns:", cat_cols)


# ============================================================
# 3.建立预处理器
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ]
)


# ============================================================
# 4.建立ExtraTrees模型
# ============================================================

extra_trees_model = ExtraTreesClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)


# ============================================================
# 5.组合Pipeline并训练
# ============================================================

extra_trees_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", extra_trees_model)
    ]
)

extra_trees_pipeline.fit(X_train, y_train)

In [ ]:
# ============================================================
# 6.验证集预测与评价指标
# ============================================================

extra_trees_pred = extra_trees_pipeline.predict(X_valid)

extra_trees_acc = accuracy_score(y_valid, extra_trees_pred)
extra_trees_balanced_acc = balanced_accuracy_score(y_valid, extra_trees_pred)
extra_trees_macro_f1 = f1_score(y_valid, extra_trees_pred, average="macro")

extra_trees_report = classification_report(
    y_valid,
    extra_trees_pred,
    zero_division=0
)

print("ExtraTrees Results")
print("=" * 60)
print(f"Accuracy: {extra_trees_acc:.6f}")
print(f"Balanced Accuracy: {extra_trees_balanced_acc:.6f}")
print(f"Macro F1: {extra_trees_macro_f1:.6f}")
print("\nClassification Report:")
print(extra_trees_report)

In [ ]:
# ============================================================
# 7.保存评价指标
# ============================================================

extra_trees_metrics = pd.DataFrame({
    "model": ["ExtraTrees_baseline"],
    "accuracy": [extra_trees_acc],
    "balanced_accuracy": [extra_trees_balanced_acc],
    "macro_f1": [extra_trees_macro_f1]
})

metrics_path = TABLE_DIR / "extra_trees_metrics.csv"
extra_trees_metrics.to_csv(metrics_path, index=False, encoding="utf-8-sig")

print("Saved metrics:", metrics_path)


# ============================================================
# 8.保存classification report为txt
# ============================================================

report_txt_path = TABLE_DIR / "extra_trees_report.txt"

with open(report_txt_path, "w", encoding="utf-8") as f:
    f.write("ExtraTrees Baseline Report\n")
    f.write("=" * 60 + "\n")
    f.write(f"Accuracy: {extra_trees_acc:.6f}\n")
    f.write(f"Balanced Accuracy: {extra_trees_balanced_acc:.6f}\n")
    f.write(f"Macro F1: {extra_trees_macro_f1:.6f}\n\n")
    f.write("Classification Report:\n")
    f.write(extra_trees_report)

print("Saved report txt:", report_txt_path)


# ============================================================
# 9.保存classification report为csv
# ============================================================

extra_trees_report_dict = classification_report(
    y_valid,
    extra_trees_pred,
    zero_division=0,
    output_dict=True
)

extra_trees_report_df = pd.DataFrame(extra_trees_report_dict).T

report_csv_path = TABLE_DIR / "extra_trees_classification_report.csv"
extra_trees_report_df.to_csv(report_csv_path, encoding="utf-8-sig")

print("Saved report csv:", report_csv_path)

In [ ]:
# ============================================================
# 10.绘制并保存混淆矩阵
# ============================================================

labels = sorted(y_valid.unique())

extra_trees_cm = confusion_matrix(
    y_valid,
    extra_trees_pred,
    labels=labels
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=extra_trees_cm,
    display_labels=labels
)

disp.plot(values_format="d")

plt.title("Confusion Matrix - ExtraTrees")
plt.savefig(
    FIGURE_DIR / "extra_trees_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

print("Saved confusion matrix figure:", FIGURE_DIR / "extra_trees_confusion_matrix.png")


# ============================================================
# 11.保存混淆矩阵为csv
# ============================================================

extra_trees_cm_df = pd.DataFrame(
    extra_trees_cm,
    index=[f"true_{label}" for label in labels],
    columns=[f"pred_{label}" for label in labels]
)

cm_csv_path = TABLE_DIR / "extra_trees_confusion_matrix.csv"
extra_trees_cm_df.to_csv(cm_csv_path, encoding="utf-8-sig")

print("Saved confusion matrix csv:", cm_csv_path)

In [ ]:
# ============================================================
# 12.保存ExtraTrees特征重要性
# ============================================================

preprocessor = extra_trees_pipeline.named_steps["preprocessor"]
extra_trees_model = extra_trees_pipeline.named_steps["model"]

num_feature_names = num_cols

cat_encoder = preprocessor.named_transformers_["cat"]
cat_feature_names = cat_encoder.get_feature_names_out(cat_cols).tolist()

all_feature_names = num_feature_names + cat_feature_names

extra_trees_feature_importance = pd.DataFrame({
    "feature": all_feature_names,
    "importance": extra_trees_model.feature_importances_
}).sort_values("importance", ascending=False)

display(extra_trees_feature_importance.head(30))


# 保存特征重要性表
feature_importance_path = TABLE_DIR / "extra_trees_feature_importance.csv"
extra_trees_feature_importance.to_csv(
    feature_importance_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved feature importance:", feature_importance_path)


# 绘制Top20特征重要性图
top_n = 20
top_features = extra_trees_feature_importance.head(top_n).sort_values("importance")

plt.figure(figsize=(10, 8))
plt.barh(top_features["feature"], top_features["importance"])
plt.title("Top 20 Feature Importance - ExtraTrees")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()

feature_importance_fig_path = FIGURE_DIR / "extra_trees_feature_importance_top20.png"
plt.savefig(feature_importance_fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved feature importance figure:", feature_importance_fig_path)

In [ ]:
# ============================================================
# Model comparison table
# ============================================================

import pandas as pd
from pathlib import Path

# 结果保存目录
comparison_dir = Path("../reports/model/baseline")
comparison_dir.mkdir(parents=True, exist_ok=True)

# 汇总Dummy、RandomForest、ExtraTrees三个模型的验证集指标
model_comparison = pd.DataFrame({
    "Model": [
        "Dummy",
        "RandomForest",
        "ExtraTrees"
    ],
    "Accuracy": [
        0.653815,
        0.956370,
        0.956188
    ],
    "Balanced Accuracy": [
        0.333333,
        0.953654,
        0.932845
    ],
    "Macro F1": [
        0.263558,
        0.942153,
        0.938546
    ]
})

# 按Macro F1从高到低排序，便于判断当前最佳模型
model_comparison = model_comparison.sort_values(
    by="Macro F1",
    ascending=False
).reset_index(drop=True)

# 保存为CSV
model_comparison.to_csv(
    comparison_dir / "model_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

# 保存为Markdown表格，可直接复制到model_report.md
with open(comparison_dir / "model_comparison.md", "w", encoding="utf-8") as f:
    f.write(model_comparison.to_markdown(index=False))

# 在notebook中显示
model_comparison